# VAE MNIST Lab — Autoencoders vs Variational Autoencoders

> Hands-on study lab built alongside `notes/09-autoencoders-and-vaes.md`.
> Framework: **PyTorch** (TensorFlow not available in this environment — noted in README).
>
> **Goal:** Train a plain Autoencoder and a VAE on MNIST, compare their latent spaces, and generate new digits from the VAE.

## 1. Setup and MNIST Loading

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LATENT_DIM = 2      # 2D so we can visualize the latent space
BATCH_SIZE = 256
EPOCHS = 10

print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

In [ ]:
# MNIST is downloaded at runtime — no dataset files are committed to the repo.
transform = transforms.Compose([
    transforms.ToTensor(),          # [0, 255] uint8 -> [0.0, 1.0] float32
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Training samples: {len(train_dataset):,}')
print(f'Test samples:     {len(test_dataset):,}')
print(f'Image shape:      {train_dataset[0][0].shape}  (C x H x W)')

In [ ]:
# Quick look at a batch of digits
sample_imgs, sample_labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_imgs[i].squeeze(), cmap='gray')
    ax.set_title(str(sample_labels[i].item()), fontsize=8)
    ax.axis('off')
plt.suptitle('Sample MNIST digits (normalized to [0, 1])', y=1.02)
plt.tight_layout()
plt.show()

Each image is a 28x28 grayscale digit, flattened to 784 values. We normalize to [0, 1] so we can use binary cross-entropy as the reconstruction loss.

---
## 2. Plain Autoencoder (latent dim = 2)

The plain autoencoder compresses 784 values down to just **2 values** in the latent space, then reconstructs the original 784.
We use 2 dimensions purely to make the latent space easy to visualize on a 2D scatter plot.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        # Encoder: 784 -> 256 -> 64 -> latent_dim
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256,  64), nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        # Decoder: latent_dim -> 64 -> 256 -> 784
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 256),        nn.ReLU(),
            nn.Linear(256, 784),       nn.Sigmoid()   # output in [0, 1]
        )

    def forward(self, x):
        z     = self.encoder(x)
        recon = self.decoder(z)
        return recon, z

ae = Autoencoder(latent_dim=LATENT_DIM).to(DEVICE)
ae_optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)
print(ae)

In [ ]:
def train_ae(model, optimizer, loader, epochs):
    model.train()
    history = []
    for epoch in range(1, epochs + 1):
        total_loss = 0
        for imgs, _ in loader:
            imgs = imgs.to(DEVICE)
            optimizer.zero_grad()
            recon, _ = model(imgs)
            # Binary cross-entropy summed over pixels (matches VAE convention)
            loss = F.binary_cross_entropy(recon, imgs.view(-1, 784), reduction='sum') / imgs.size(0)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg = total_loss / len(loader)
        history.append(avg)
        print(f'  AE Epoch {epoch:02d}/{epochs}  loss={avg:.2f}')
    return history

print('Training plain Autoencoder...')
ae_history = train_ae(ae, ae_optimizer, train_loader, EPOCHS)

In [ ]:
# Show original vs reconstructed digits for the AE
ae.eval()
with torch.no_grad():
    sample, _ = next(iter(test_loader))
    sample = sample.to(DEVICE)
    recon, _ = ae(sample)

n = 8
fig, axes = plt.subplots(2, n, figsize=(12, 3))
for i in range(n):
    axes[0, i].imshow(sample[i].cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(recon[i].cpu().view(28, 28), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=9)
axes[1, 0].set_ylabel('AE Recon', fontsize=9)
plt.suptitle('Autoencoder: Original vs Reconstructed')
plt.tight_layout()
plt.show()

The reconstructions are recognizable but somewhat blurry — this is expected with a tiny 2D latent space. The plain AE is only optimizing reconstruction; it does not care about the structure of the 2D space.

---
## 3. Variational Autoencoder (latent dim = 2)

The VAE encoder outputs **two vectors**: μ (mean) and log σ² (log variance).
We use the **reparameterization trick** to sample z = μ + σ·ε where ε ~ N(0, 1).

We output log σ² (z_log_var) rather than σ directly for numerical stability.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder shared layers
        self.enc_shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256,  64), nn.ReLU(),
        )
        # Two output heads: one for μ, one for log σ²
        self.fc_mu      = nn.Linear(64, latent_dim)
        self.fc_log_var = nn.Linear(64, latent_dim)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 256),        nn.ReLU(),
            nn.Linear(256, 784),       nn.Sigmoid()
        )

    def encode(self, x):
        h       = self.enc_shared(x)
        mu      = self.fc_mu(h)
        log_var = self.fc_log_var(h)
        return mu, log_var

    def reparameterize(self, mu, log_var):
        # z = mu + sigma * epsilon,  epsilon ~ N(0, 1)
        # sigma = exp(0.5 * log_var)
        sigma = torch.exp(0.5 * log_var)
        eps   = torch.randn_like(sigma)   # sample epsilon
        return mu + sigma * eps

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z           = self.reparameterize(mu, log_var)
        recon       = self.decode(z)
        return recon, mu, log_var


def vae_loss(recon, x, mu, log_var):
    # Reconstruction loss: binary cross-entropy summed over pixels, averaged over batch
    recon_loss = F.binary_cross_entropy(recon, x.view(-1, 784), reduction='sum') / x.size(0)
    # KL divergence: D_KL( N(mu, sigma^2) || N(0, I) ) = -0.5 * sum(1 + log_var - mu^2 - exp(log_var))
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)
    # VAE loss = Reconstruction loss + KL  (this is -ELBO, which we minimize)
    return recon_loss + kl_loss, recon_loss, kl_loss


vae = VAE(latent_dim=LATENT_DIM).to(DEVICE)
vae_optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
print(vae)

In [ ]:
def train_vae(model, optimizer, loader, epochs):
    model.train()
    history = {'total': [], 'recon': [], 'kl': []}
    for epoch in range(1, epochs + 1):
        tot, rec, kl_sum = 0, 0, 0
        for imgs, _ in loader:
            imgs = imgs.to(DEVICE)
            optimizer.zero_grad()
            recon, mu, log_var = model(imgs)
            loss, recon_loss, kl_loss = vae_loss(recon, imgs, mu, log_var)
            loss.backward()
            optimizer.step()
            tot    += loss.item()
            rec    += recon_loss.item()
            kl_sum += kl_loss.item()
        n = len(loader)
        history['total'].append(tot / n)
        history['recon'].append(rec / n)
        history['kl'].append(kl_sum / n)
        print(f'  VAE Epoch {epoch:02d}/{epochs}  total={tot/n:.2f}  recon={rec/n:.2f}  kl={kl_sum/n:.2f}')
    return history

print('Training VAE...')
vae_history = train_vae(vae, vae_optimizer, train_loader, EPOCHS)

---
## 4. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# AE loss
axes[0].plot(ae_history, color='steelblue', linewidth=2)
axes[0].set_title('Autoencoder Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (BCE)')
axes[0].grid(alpha=0.3)

# VAE losses
axes[1].plot(vae_history['total'], label='Total',         color='darkorange', linewidth=2)
axes[1].plot(vae_history['recon'], label='Reconstruction', color='steelblue',  linewidth=2, linestyle='--')
axes[1].plot(vae_history['kl'],    label='KL',             color='crimson',    linewidth=2, linestyle=':')
axes[1].set_title('VAE Training Losses')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Training curves — AE vs VAE', fontsize=13)
plt.tight_layout()
plt.show()

The VAE total loss is a sum of reconstruction loss and KL divergence. As training progresses the KL term rises (the model learns a broader but structured latent distribution) while reconstruction loss falls. The plain AE just minimizes reconstruction with no latent-space regularization.

---
## 5. Latent-Space Scatter Plots — AE vs VAE

We encode the entire test set and plot each digit in the 2D latent space, colored by class label.

In [ ]:
def encode_all(model, loader, model_type='ae'):
    model.eval()
    zs, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(DEVICE)
            if model_type == 'ae':
                _, z = model(imgs)
            else:  # vae
                mu, _ = model.encode(imgs)
                z = mu   # use the mean for visualization
            zs.append(z.cpu().numpy())
            labels.append(lbls.numpy())
    return np.concatenate(zs), np.concatenate(labels)

ae_z,  ae_labels  = encode_all(ae,  test_loader, 'ae')
vae_z, vae_labels = encode_all(vae, test_loader, 'vae')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = plt.cm.get_cmap('tab10', 10)

for digit in range(10):
    mask = ae_labels == digit
    axes[0].scatter(ae_z[mask, 0], ae_z[mask, 1], s=2, alpha=0.5, color=cmap(digit), label=str(digit))
axes[0].set_title('AE latent space — notice scattered clusters / gaps')
axes[0].set_xlabel('z[0]');  axes[0].set_ylabel('z[1]')
axes[0].legend(markerscale=4, loc='upper right', title='Digit', fontsize=7)
axes[0].grid(alpha=0.2)

for digit in range(10):
    mask = vae_labels == digit
    axes[1].scatter(vae_z[mask, 0], vae_z[mask, 1], s=2, alpha=0.5, color=cmap(digit), label=str(digit))
axes[1].set_title('VAE latent space — compact, centered, continuous')
axes[1].set_xlabel('z[0]');  axes[1].set_ylabel('z[1]')
axes[1].legend(markerscale=4, loc='upper right', title='Digit', fontsize=7)
axes[1].grid(alpha=0.2)

plt.suptitle('Latent space: AE vs VAE (test set, colored by digit)', fontsize=13)
plt.tight_layout()
plt.show()

The AE latent space has clusters scattered at arbitrary positions with empty gaps between them. If you sample a random point in one of those gaps and decode it, you get noise — there is no guarantee of a meaningful output.

The VAE latent space is compact and centered around (0, 0). The clusters for different digits overlap more at the boundaries, reflecting the smooth transitions forced by the KL regularization. Sampling any point near the origin is more likely to give a meaningful digit.

---
## 6. Generating New Digits — Sample z ~ N(0, I)

Since the VAE learns a structured latent space, we can sample random z vectors from N(0, I) and decode them into new digits.

In [ ]:
# Sample random z vectors from N(0, I) and decode
vae.eval()
with torch.no_grad():
    z_samples = torch.randn(16, LATENT_DIM).to(DEVICE)
    generated  = vae.decode(z_samples).cpu().view(-1, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i], cmap='gray')
    ax.axis('off')
plt.suptitle('VAE: New digits generated by sampling z ~ N(0, I)')
plt.tight_layout()
plt.show()

In [ ]:
# 15x15 latent grid walk — sweep z across a uniform grid from -3 to +3
vae.eval()
n_grid = 15
grid_range = np.linspace(-3, 3, n_grid)

canvas = np.zeros((28 * n_grid, 28 * n_grid))
with torch.no_grad():
    for i, y in enumerate(grid_range):
        for j, x in enumerate(grid_range):
            z = torch.tensor([[x, y]], dtype=torch.float32).to(DEVICE)
            img = vae.decode(z).cpu().view(28, 28).numpy()
            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = img

plt.figure(figsize=(8, 8))
plt.imshow(canvas, cmap='gray')
plt.axis('off')
plt.title('VAE latent space grid walk (z[0] from -3 to +3, z[1] from -3 to +3)')
plt.tight_layout()
plt.show()

The 15x15 grid shows how the decoder output transitions smoothly as we move through the latent space. This smooth interpolation is the key advantage of the VAE — its KL regularization ensures the latent space is continuous and organized, so every point in it decodes to something meaningful. Walking from one cluster to another, you can see the digit gradually morph from one shape to another.

---
## Summary

| Aspect | Autoencoder | VAE |
|---|---|---|
| Latent space structure | Unstructured — gaps exist | Compact, continuous, centered at N(0, I) |
| Random sampling | May produce noise from empty regions | Works reliably — any N(0, I) sample is meaningful |
| Loss | Reconstruction only | Reconstruction + KL divergence |
| Reparameterization | Not needed | Required to allow gradient flow through the sampling step |

The key takeaway: the VAE's KL term forces the learned latent distribution toward a standard Gaussian, filling in the gaps and making the space smooth enough for controlled generation.

**Next:** see `notes/09-autoencoders-and-vaes.md` for the full theory behind everything shown here.